# Corrective RAG (CRAG) with LangGraph
## Retrieve, Grade, Rewrite — The CRAG Correction Loop
⏱ ~15 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/17-corrective-rag/corrective_rag_workbook.ipynb)

**Standard RAG has a silent failure mode:** it retrieves documents and generates an answer regardless of whether those documents are actually relevant. If your vectorstore contains stale data, the wrong topic, or simply no matching content, the LLM hallucinates anyway.

**Corrective RAG (CRAG)** fixes this by adding a self-evaluation loop between retrieval and generation. Each retrieved document is graded by an LLM. If any document fails the relevance check, the pipeline rewrites the query and falls back to a live web search before generating — replacing bad local context with fresh, targeted information.

This workbook implements the full CRAG pipeline from [Yan et al. (2024)](https://arxiv.org/abs/2401.15884) using **LangGraph** — a graph-based framework for building stateful, multi-step LLM pipelines.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Standard RAG vs CRAG vs Self-RAG |
| 2 | **Setup** — Install, API key, imports |
| 3 | **Knowledge Base** — Build the Chroma vectorstore |
| 4 | **LLM Grader** — `with_structured_output` for yes/no scoring |
| 5 | **CRAG Graph** — Five nodes wired with conditional edges |
| 6 | **Running the Pipeline** — In-corpus and out-of-corpus queries |
| 7 | **Streaming** — Node-by-node observability |
| 8 | **Debugging** — Inspect grading decisions and state |
| ★ | **Exercises + Answer Key** |

---

### Prerequisites
- Python 3.10+ with a `.venv` containing the repo's `requirements.txt`
- An `OPENAI_API_KEY` in `.env` (or Colab Secrets)
- Internet access for web search fallback (DuckDuckGo — no API key needed)

### Key References
> Yan, S., Gu, J., Zhu, Y., & Ling, Z. (2024). *Corrective Retrieval Augmented Generation.* arXiv:2401.15884. https://arxiv.org/abs/2401.15884
> Lewis, P., Perez, E., et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.* NeurIPS 2020. https://arxiv.org/abs/2005.11401
> LangGraph documentation: https://langchain-ai.github.io/langgraph/

---

## Part 1 — Concepts: Standard RAG vs CRAG vs Self-RAG

---

### The Problem with Standard RAG

Standard RAG retrieves top-k documents by vector similarity and passes them to the LLM — every time, unconditionally. There is no quality gate. If your vectorstore returns a tangentially related document, or a document from the wrong domain, the LLM still uses it. The result is a subtly wrong, confidently stated answer.

CRAG adds **self-evaluation**: an LLM judges each retrieved document before it reaches the generator. This creates a corrective feedback loop that maintains answer quality even when the local corpus is incomplete or mismatched.

---

### Three Approaches Compared

| Approach | Self-evaluation | Web fallback | Complexity | When to use |
|----------|----------------|--------------|------------|-------------|
| **Standard RAG** | None | Never | Low | Stable, well-curated corpus |
| **CRAG** | Per-document grading | On low-quality retrieval | Medium | Mixed-quality corpus, fresh data needed |
| **Self-RAG** | Token-level reflection tokens | Optional | High | Maximum control over generation quality |

**CRAG is the practical middle ground**: it adds meaningful self-correction without the complexity of full reflection tokens or multi-turn critique loops.

---

### The CRAG Flow

```
User query
    |
    v
+-----------+
|  retrieve  |   top-k documents from Chroma vectorstore
+-----+-----+
      |
      v
+---------------+
|grade_documents|   LLM scores each doc: 'yes' (relevant) or 'no' (irrelevant)
+-------+-------+
        |
        +--- ALL docs relevant ----------------------------------+
        |                                                        |
        +--- ANY doc irrelevant ---> +-----------------+        |
                                     | transform_query  |        |
                                     +--------+--------+        |
                                              |                  |
                                              v                  |
                                     +----------------+          |
                                     | web_search_node|          |
                                     +--------+-------+          |
                                              |                  |
                                              v                  |
                                     +-------------+ <-----------+
                                     |  generate   |
                                     +------+------+
                                            |
                                            v
                                         Answer
```

> Source: Adapted from Yan et al. (2024) CRAG paper, Figure 1.

---

### Key Vocabulary

| Term | Definition |
|------|------------|
| **Grader** | An LLM call that returns a structured yes/no score for a retrieved document |
| **Conditional edge** | A LangGraph routing function that chooses the next node based on state |
| **Query rewriting** | Rephrasing the original question to improve web search recall |
| **StateGraph** | LangGraph's typed state machine — each node reads and updates a shared state dict |
| **`with_structured_output`** | LangChain method that forces an LLM to return a Pydantic model |
| **Web fallback** | Replacing low-quality local context with live search results |
| **Correction trigger** | The condition (`web_search: True`) that activates the fallback path |

---

## Part 2 — Setup: Install, API Key, Imports

---

### 2-A: Install dependencies (Colab only)

Local users already have everything from `requirements.txt`. The `_in_colab()` helper detects the environment so this cell is safe to run anywhere.

In [ ]:
# ─── 2-A: Install dependencies — runs only on Google Colab ───────────────────
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "langchain==0.3.27",
            "langchain-core==0.3.79",
            "langchain-openai==0.3.33",
            "langchain-chroma==0.2.6",
            "langchain-community==0.3.31",
            "langchain-text-splitters==0.3.11",
            "langgraph==0.6.7",
            "duckduckgo-search==8.1.1",
            "chromadb==1.1.0",
            "python-dotenv",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

### 2-B: Load the API key

On **Colab**: open the key icon in the left sidebar → add a secret named `OPENAI_API_KEY`.  
On **local**: create a `.env` file in the project root containing `OPENAI_API_KEY=sk-...`

The cell tries Colab Secrets first, then falls back to `python-dotenv`.

In [ ]:
# ─── 2-B: Cross-platform API key ─────────────────────────────────────────────
import os

try:
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv

    load_dotenv()

key = os.getenv("OPENAI_API_KEY", "")
assert key.startswith("sk-"), (
    "OPENAI_API_KEY not found or invalid. "
    "Local: add 'OPENAI_API_KEY=sk-...' to .env | "
    "Colab: set it in the Secrets panel (key icon in sidebar)"
)
print(f"OPENAI_API_KEY loaded: {key[:8]}...{key[-4:]}")

In [ ]:
# ─── 2-C: Core imports ───────────────────────────────────────────────────────
from typing import Literal, TypedDict

from langchain_chroma import Chroma
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel

print("All imports successful.")

---

## Part 3 — Knowledge Base: Build the Chroma Vectorstore

---

The CRAG pipeline starts with a local vectorstore. In production this would contain your internal documents — PDFs, docs pages, knowledge base articles. For this workshop we use five in-memory documents covering LangGraph, CRAG, and RAG concepts.

**Why this corpus size matters for learning:** It is small enough that out-of-corpus queries (about current events or unrelated topics) will clearly fail retrieval and trigger the web fallback. You will see both paths of the pipeline in action.

### Retrieval parameters

| Parameter | Value | Rationale |
|-----------|-------|----------|
| `chunk_size` | 500 | Each document fits in one chunk — avoids splitting mid-sentence |
| `chunk_overlap` | 50 | Ensures continuity if a document were split |
| `k` | 3 | Top-3 documents retrieved per query |
| Embedding model | `text-embedding-3-small` | Cost-effective, 1536 dimensions |

In [ ]:
# ─── 3-A: Define the document corpus ─────────────────────────────────────────
CORPUS = [
    Document(
        page_content=(
            "LangGraph is a library for building stateful, multi-actor applications "
            "with LLMs. It extends LangChain with the ability to coordinate multiple "
            "chains in a graph of nodes and edges, supporting cycles, branching, and "
            "checkpointing."
        ),
        metadata={"source": "langgraph-docs", "topic": "langgraph"},
    ),
    Document(
        page_content=(
            "Corrective RAG (CRAG) improves retrieval by grading each retrieved document "
            "for relevance. Irrelevant documents trigger a query rewrite followed by a "
            "web search, replacing low-quality local context with fresh information."
        ),
        metadata={"source": "crag-paper", "topic": "crag"},
    ),
    Document(
        page_content=(
            "RAG (Retrieval Augmented Generation) grounds LLM responses in a knowledge "
            "base. A retriever fetches relevant documents; those documents become the "
            "context an LLM uses to generate a factual, sourced answer."
        ),
        metadata={"source": "rag-paper", "topic": "rag"},
    ),
    Document(
        page_content=(
            "LangGraph StateGraph lets you define typed state, add nodes as Python "
            "functions, and wire them with edges and conditional edges. The compiled "
            "graph handles state transitions, streaming, and optional persistence via "
            "checkpointers."
        ),
        metadata={"source": "langgraph-docs", "topic": "stategraph"},
    ),
    Document(
        page_content=(
            "ChromaDB is an open-source embedding database. LangChain's Chroma "
            "integration stores document embeddings locally and supports "
            "cosine-similarity retrieval via the as_retriever() interface."
        ),
        metadata={"source": "chroma-docs", "topic": "chromadb"},
    ),
]

print(f"Corpus: {len(CORPUS)} documents")
for doc in CORPUS:
    print(f"  [{doc.metadata['topic']:12}] {doc.page_content[:70]}...")

In [ ]:
# ─── 3-B: Build the Chroma vectorstore and retriever ─────────────────────────
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(CORPUS)

vectorstore = Chroma.from_documents(
    chunks,
    embeddings_model,
    collection_name="crag-workbook",
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"Vectorstore ready: {len(chunks)} chunks indexed")
print()

# Sanity check: verify retrieval works before building the graph
test_docs = retriever.invoke("What is Corrective RAG?")
print(f"Retrieval test 'What is Corrective RAG?' -> {len(test_docs)} docs returned")
for i, doc in enumerate(test_docs, 1):
    print(f"  [{i}] topic={doc.metadata.get('topic', '?')}: {doc.page_content[:80]}...")

---

## Part 4 — LLM Grader: Structured Yes/No Scoring

---

The grader is the heart of CRAG. It asks the LLM a binary question: *Does this document directly help answer the user's question?* The LLM must return a structured `yes` or `no` — not free text.

### `with_structured_output` — the modern LangChain pattern

Instead of parsing free-text LLM responses with regex or string splitting, `with_structured_output` forces the model to return a **Pydantic model**. This eliminates parsing errors and makes grading deterministic.

```python
class GradeDocuments(BaseModel):
    score: Literal["yes", "no"]

grader = llm.with_structured_output(GradeDocuments)
# grader.invoke(...) always returns GradeDocuments, never a raw string
```

Under the hood, LangChain uses function calling / tool use to constrain the output schema.

### Three separate prompts, three responsibilities

| Prompt | Purpose | Output type |
|--------|---------|-------------|
| `grade_prompt` | Evaluate document relevance | `GradeDocuments(score='yes'/'no')` |
| `answer_prompt` | Generate the final grounded answer | Plain text string |
| `rewrite_prompt` | Rephrase query for web search | Plain text string (improved query) |

Each prompt has a single, narrow responsibility — making them easy to tune independently.

In [ ]:
# ─── 4-A: Define the grader model and all three prompts ──────────────────────


class GradeDocuments(BaseModel):
    """Binary relevance score for a retrieved document."""

    score: Literal["yes", "no"]


llm = ChatOpenAI(model="gpt-5-nano", temperature=0)
grader = llm.with_structured_output(GradeDocuments)

# Grading prompt: require direct relevance, not just topic overlap
grade_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a document relevance grader. "
            "Given a question and a document, decide if the document contains "
            "information that DIRECTLY helps answer the question. "
            "Answer 'yes' only if the document is genuinely useful. "
            "Answer 'no' if it is only tangentially related or off-topic.",
        ),
        ("human", "Question: {question}\n\nDocument:\n{document}"),
    ]
)

# Generation prompt: grounded answer, no hallucination
answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Answer the user's question using ONLY the information in the context below. "
            "If the context does not contain enough information, say "
            "'I do not have enough information to answer this question.' "
            "Do not make up details.\n\nContext:\n{context}",
        ),
        ("human", "{question}"),
    ]
)

# Rewrite prompt: improve specificity for web search
rewrite_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a search query optimizer. "
            "Rewrite the given question to be more specific, factual, and web-searchable. "
            "Include key terms that would appear in relevant web pages. "
            "Return only the rewritten question, nothing else.",
        ),
        ("human", "{question}"),
    ]
)

print("LLM, grader, and all three prompts defined.")

In [ ]:
# ─── 4-B: Grader smoke test — verify structured output before building the graph

test_question = "What is Corrective RAG?"

test_cases = [
    (
        "Relevant (should score 'yes')",
        "CRAG improves retrieval by grading each document for relevance before generating.",
    ),
    (
        "Irrelevant (should score 'no')",
        "The 2024 Summer Olympics took place in Paris, France in July and August.",
    ),
    (
        "Tangentially related (borderline)",
        "RAG grounds LLM responses in a knowledge base using retrieved documents.",
    ),
]

print(f"Grading documents for: '{test_question}'\n")
for label, doc_text in test_cases:
    result = (grade_prompt | grader).invoke(
        {"question": test_question, "document": doc_text}
    )
    marker = "[PASS]" if result.score == "yes" else "[FILTER]"
    print(f"  {marker} {label}")
    print(f"         score = {result.score!r}")
    print(f"         doc   = {doc_text[:80]}...")
    print()

In [ ]:
# ─── 4-C: Query rewrite smoke test ────────────────────────────────────────────
# The rewrite step turns a vague user question into a specific, search-friendly query.

vague_questions = [
    "Who won?",
    "What's the latest Python?",
    "Tell me about AI stuff from last month",
]

print("Query rewriting demo:\n")
for q in vague_questions:
    rewritten = (rewrite_prompt | llm).invoke({"question": q})
    print(f"  Original:  {q}")
    print(f"  Rewritten: {rewritten.content}")
    print()

---

## Part 5 — The CRAG Graph: Five Nodes, One Conditional Edge

---

### LangGraph fundamentals

LangGraph models an agent as a **directed graph**. Each node is a Python function that receives the current state and returns a partial state update. Edges route between nodes — either unconditionally (`add_edge`) or based on the state (`add_conditional_edges`).

```
START
  |
retrieve
  |
grade_documents ---(conditional)---+--> generate --> END
                                   |
                       transform_query --> web_search_node --> generate --> END
```

### `CRAGState` — the shared state bag

| Field | Type | Updated by |
|-------|------|------------|
| `question` | `str` | `transform_query` (rewritten), initial input |
| `documents` | `list` | `retrieve`, `grade_documents` (filtered), `web_search_node` (augmented) |
| `generation` | `str` | `generate` |
| `web_search` | `bool` | `grade_documents` — set to `True` if any doc is irrelevant |

Every node returns **only the fields it changes** — not the full state. LangGraph merges the partial update into the running state automatically.

### Node responsibilities

| Node | Reads from state | Writes to state | Side effect |
|------|-----------------|-----------------|-------------|
| `retrieve` | `question` | `documents`, `web_search=False` | Chroma similarity search |
| `grade_documents` | `question`, `documents` | `documents` (filtered), `web_search` | LLM grading (one call per doc) |
| `transform_query` | `question` | `question` (rewritten) | LLM rewrite call |
| `web_search_node` | `question`, `documents` | `documents` (augmented with web results) | DuckDuckGo search |
| `generate` | `question`, `documents` | `generation` | LLM generation call |

In [ ]:
# ─── 5-A: Define the typed state and a helper constructor ─────────────────────


class CRAGState(TypedDict):
    """Shared state bag passed through every node in the CRAG graph."""

    question: str  # original (or rewritten) user question
    documents: list  # retrieved and/or web-search docs
    generation: str  # final LLM answer
    web_search: bool  # True = at least one document was graded irrelevant


def make_initial_state(question: str) -> CRAGState:
    """Create a clean initial state for a new query."""
    return {"question": question, "documents": [], "generation": "", "web_search": False}


print("CRAGState defined with fields:", list(CRAGState.__annotations__.keys()))

In [ ]:
# ─── 5-B: Define the five graph nodes ────────────────────────────────────────

web_search_tool = DuckDuckGoSearchResults(max_results=3)


def retrieve(state: CRAGState) -> dict:
    """Fetch top-k documents from the Chroma vectorstore."""
    docs = retriever.invoke(state["question"])
    return {"documents": docs, "web_search": False}


def grade_documents(state: CRAGState) -> dict:
    """Grade each document. Filter irrelevant docs and set the web_search flag."""
    relevant_docs = []
    needs_web = False

    for doc in state["documents"]:
        result = (grade_prompt | grader).invoke(
            {"question": state["question"], "document": doc.page_content}
        )
        if result.score == "yes":
            relevant_docs.append(doc)
        else:
            needs_web = True  # at least one doc failed — activate fallback

    return {"documents": relevant_docs, "web_search": needs_web}


def transform_query(state: CRAGState) -> dict:
    """Rewrite the question to improve web search recall."""
    rewritten = (rewrite_prompt | llm).invoke({"question": state["question"]})
    return {"question": rewritten.content}


def web_search_node(state: CRAGState) -> dict:
    """Run DuckDuckGo search and append results to the document list."""
    results = web_search_tool.invoke(state["question"])
    web_docs = [Document(page_content=str(results), metadata={"source": "web-search"})]
    # Merge: keep any locally-relevant docs that passed grading + add fresh web results
    return {"documents": state["documents"] + web_docs}


def generate(state: CRAGState) -> dict:
    """Generate the final answer from the (possibly corrected) context."""
    context = "\n\n".join(d.page_content for d in state["documents"])
    if not context.strip():
        return {"generation": "I do not have enough information to answer this question."}
    response = (answer_prompt | llm).invoke(
        {"context": context, "question": state["question"]}
    )
    return {"generation": response.content}


def decide_after_grading(state: CRAGState) -> Literal["transform_query", "generate"]:
    """Routing function: activates web search path or proceeds directly to generation."""
    return "transform_query" if state["web_search"] else "generate"


print("All 5 nodes and routing function defined.")

In [ ]:
# ─── 5-C: Assemble and compile the CRAG graph ────────────────────────────────

graph = StateGraph(CRAGState)

# Register nodes
graph.add_node("retrieve", retrieve)
graph.add_node("grade_documents", grade_documents)
graph.add_node("transform_query", transform_query)
graph.add_node("web_search_node", web_search_node)
graph.add_node("generate", generate)

# Wire edges — unconditional
graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "grade_documents")

# Wire the conditional edge: decide_after_grading chooses the next node
graph.add_conditional_edges(
    "grade_documents",
    decide_after_grading,
    # Explicit mapping documents the two possible routes
    {"transform_query": "transform_query", "generate": "generate"},
)

# Complete the web search path
graph.add_edge("transform_query", "web_search_node")
graph.add_edge("web_search_node", "generate")
graph.add_edge("generate", END)

app = graph.compile()
print("CRAG graph compiled successfully.")
print()
print(app.get_graph().draw_mermaid())

In [ ]:
# ─── 5-D: Visualise the compiled graph ───────────────────────────────────────
# Renders as a PNG in Colab and JupyterLab. Falls back to Mermaid text locally.

try:
    from IPython.display import Image, display

    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"PNG render unavailable ({e})")
    print("Mermaid source — paste at mermaid.live to visualise:")
    print(app.get_graph().draw_mermaid())

---

## Part 6 — Running the Pipeline

---

We test two distinct types of queries:

- **In-corpus** — questions about LangGraph, CRAG, and ChromaDB. The vectorstore has good answers. Expect `web_search: False`.
- **Out-of-corpus** — questions about current events or unrelated topics. Retrieval quality will be low, the grader will reject documents, and the pipeline falls back to web search. Expect `web_search: True`.

The `web_search` flag is the primary diagnostic signal for which path the pipeline took.

In [ ]:
# ─── 6-A: In-corpus queries (expect web_search=False) ────────────────────────

in_corpus_queries = [
    "What is Corrective RAG and how does it improve retrieval?",
    "How does LangGraph StateGraph handle conditional branching?",
    "What is ChromaDB and how does it integrate with LangChain?",
]

print("=== IN-CORPUS QUERIES (vectorstore should handle these) ===\n")
for q in in_corpus_queries:
    result = app.invoke(make_initial_state(q))
    path = "LOCAL" if not result["web_search"] else "WEB FALLBACK"
    print(f"Q: {q}")
    print(f"   Path: {path}  |  Docs used: {len(result['documents'])}")
    print(f"   A: {result['generation'][:200]}")
    print()

In [ ]:
# ─── 6-B: Out-of-corpus queries (expect web_search=True) ─────────────────────

out_of_corpus_queries = [
    "Who won the 2024 US presidential election?",
    "What is the latest stable Python version as of 2025?",
]

print("=== OUT-OF-CORPUS QUERIES (should trigger web search fallback) ===\n")
for q in out_of_corpus_queries:
    result = app.invoke(make_initial_state(q))
    path = "LOCAL" if not result["web_search"] else "WEB FALLBACK"
    print(f"Q: {q}")
    print(f"   Path: {path}  |  Docs used: {len(result['documents'])}")
    print(f"   A: {result['generation'][:200]}")
    print()

---

## Part 7 — Streaming: Node-by-Node Observability

---

One of LangGraph's most useful features is **streaming**. Instead of waiting for the full pipeline to finish, `.stream(stream_mode='updates')` emits a dict after each node completes. You can watch the CRAG pipeline unfold step by step in real time.

This is invaluable for debugging: you can see exactly when documents get filtered, when the `web_search` flag flips, and what the rewritten query looks like — all before generation runs.

### Stream modes compared

| Mode | What it emits | Tokens per event |
|------|---------------|------------------|
| `"updates"` | Partial state changes from each node `{node_name: updates}` | Low |
| `"values"` | Full state snapshot after each node | High |
| `"debug"` | Task dispatches, state snapshots, errors — everything | Very high |

In [ ]:
# ─── 7-A: Stream an in-corpus query — watch the local path ───────────────────

question = "What are the key differences between CRAG and standard RAG?"
print(f'Streaming CRAG pipeline for: "{question}"')
print("=" * 60)

for step in app.stream(make_initial_state(question), stream_mode="updates"):
    node_name = list(step.keys())[0]
    updates = step[node_name]
    print(f"\n[{node_name}]")
    if "documents" in updates:
        print(f"  docs in state: {len(updates['documents'])}")
        for doc in updates["documents"]:
            src = doc.metadata.get("source", "?")
            print(f"    [{src}] {doc.page_content[:60]}...")
    if "web_search" in updates:
        print(f"  web_search flag: {updates['web_search']}")
    if "question" in updates:
        print(f"  question (rewritten): {updates['question']}")
    if "generation" in updates:
        print(f"  answer: {updates['generation'][:200]}")

In [ ]:
# ─── 7-B: Stream an out-of-corpus query — watch the web fallback path ─────────

question_oc = "What new AI models were released in early 2025?"
print(f'Streaming CRAG pipeline for: "{question_oc}"')
print("=" * 60)

for step in app.stream(make_initial_state(question_oc), stream_mode="updates"):
    node_name = list(step.keys())[0]
    updates = step[node_name]
    print(f"\n[{node_name}]")
    if "documents" in updates:
        print(f"  docs in state: {len(updates['documents'])}")
    if "web_search" in updates:
        flag = updates["web_search"]
        indicator = "TRIGGERED" if flag else "not triggered"
        print(f"  web_search: {flag}  ({indicator})")
    if "question" in updates:
        print(f"  rewritten query: {updates['question']}")
    if "generation" in updates:
        print(f"  answer: {updates['generation'][:200]}")

---

## Part 8 — Debugging: Inspect Grading Decisions and State

---

### Common CRAG failure modes

| Failure | Symptom | Root cause | Fix |
|---------|---------|-----------|-----|
| **Over-filtering** | Web search fires even for clearly relevant docs | Grade prompt too strict | Soften the grader system prompt |
| **Under-filtering** | Irrelevant docs pass grading | Grade prompt too lenient | Add: "only yes if it directly answers" |
| **Rewrite misfire** | Rewritten query is worse than original | Rewrite prompt too generic | Add domain context to the rewrite prompt |
| **Web search noise** | Web results unrelated to the question | Rewritten query still vague | Require more specificity in rewrite prompt |
| **Empty context** | Generation fallback fires too often | All docs graded `no` and web search fails | Expand corpus or lower `k` |

### Debugging checklist

1. Run `debug_grading()` (cell 8-A) before invoking the graph — see individual doc scores
2. Use `stream_mode='updates'` (Part 7) to trace node-by-node state changes
3. Run the grader standalone on specific docs to tune the prompt text
4. Print the rewritten query — it should look like a good web search query

In [ ]:
# ─── 8-A: Debug grading decisions for any query ───────────────────────────────
# Run this before graph.invoke() to understand WHY documents pass or fail.


def debug_grading(question: str) -> bool:
    """Retrieve docs and show individual grading decisions. Returns web_search flag."""
    docs = retriever.invoke(question)
    print(f"Query: '{question}'")
    print(f"Retrieved {len(docs)} docs from vectorstore:\n")

    relevant_count = 0
    for i, doc in enumerate(docs, 1):
        result = (grade_prompt | grader).invoke(
            {"question": question, "document": doc.page_content}
        )
        status = "RELEVANT" if result.score == "yes" else "FILTERED"
        if result.score == "yes":
            relevant_count += 1
        src = doc.metadata.get("source", "?")
        print(f"  [{i}] {status} (score={result.score!r})")
        print(f"       source: {src}")
        print(f"       content: {doc.page_content[:100]}...")
        print()

    needs_web = relevant_count < len(docs)
    decision = "WEB FALLBACK triggered" if needs_web else "local context sufficient"
    print(f"Decision: {decision} ({relevant_count}/{len(docs)} docs passed)")
    return needs_web


print("=" * 60)
debug_grading("How does LangGraph handle state?")
print("\n" + "=" * 60)
debug_grading("What is the population of Tokyo in 2025?")

In [ ]:
# ─── 8-B: Debug the query rewrite step ───────────────────────────────────────
# A good rewritten query should look like something you would type into Google.

queries_to_inspect = [
    "Who won?",
    "What happened recently in AI?",
    "Latest news about Python",
    "Tell me about that new model from Anthropic",
]

print("Query rewrite diagnostic:\n")
print(f"{'ORIGINAL':<45} REWRITTEN")
print("-" * 110)
for q in queries_to_inspect:
    rewritten = (rewrite_prompt | llm).invoke({"question": q})
    print(f"{q:<45} {rewritten.content}")

---

## Exercises

---

Work through these in order — each one builds on the previous. The answer key is at the bottom of this notebook. Attempt each exercise before reading the solution.

### Exercise 1 — Stricter grader

The current grade prompt accepts documents that relate to the same general topic. Rewrite `grade_prompt` so the grader only scores `yes` if the document **directly contains a fact that answers the specific question** — not just overlapping keywords.

**Goal:** The borderline RAG document from 4-B (which describes RAG in general but does not explain CRAG's correction mechanism) should now score `no`.

### Exercise 2 — Confidence threshold

Extend `GradeDocuments` with a `confidence: float` field (0.0 to 1.0). Update `grade_documents` to only keep documents where `score == 'yes'` AND `confidence >= 0.7`. A document that scores `yes` with `confidence=0.5` should be treated the same as a `no`.

### Exercise 3 — Expand the corpus

Add five new documents on a topic of your choice (e.g. ReAct agents, LLM evaluation, vector databases) to `CORPUS`. Rebuild the vectorstore from Part 3. Verify that questions on your new topic no longer trigger web search.

### Exercise 4 — Persistent checkpointing

Recompile the graph with a `MemorySaver` checkpointer so state is preserved across invocations on the same thread. Invoke twice with the same `thread_id`. Inspect the saved state history.

```python
from langgraph.checkpoint.memory import MemorySaver
app = graph.compile(checkpointer=MemorySaver())
config = {"configurable": {"thread_id": "workshop-demo"}}
```

In [ ]:
# ── Exercise 1 Starter ── Stricter grader ─────────────────────────────────────

strict_grade_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            # TODO: rewrite this system prompt so that only documents containing
            # a DIRECT FACTUAL ANSWER to the question score 'yes'.
            # Keyword overlap alone should NOT be sufficient for a 'yes'.
            "TODO: write a stricter grading instruction here",
        ),
        ("human", "Question: {question}\n\nDocument:\n{document}"),
    ]
)

# Test on the borderline case from 4-B
borderline_doc = "RAG grounds LLM responses in a knowledge base using retrieved documents."
test_q = "What is Corrective RAG?"

# TODO: invoke (strict_grade_prompt | grader) with the borderline doc
# Expected: score='no' because this doc describes RAG but not CRAG specifically
# result = (strict_grade_prompt | grader).invoke({"question": test_q, "document": borderline_doc})
# print(f"Borderline doc score: {result.score!r}  (expected: 'no')")

In [ ]:
# ── Exercise 2 Starter ── Confidence threshold ────────────────────────────────


class GradeDocumentsWithConfidence(BaseModel):
    """Document relevance score with a confidence estimate."""

    score: Literal["yes", "no"]
    confidence: float  # TODO: constrain to 0.0-1.0 with pydantic Field


CONFIDENCE_THRESHOLD = 0.7

# TODO: create a new grader using llm.with_structured_output(GradeDocumentsWithConfidence)
# TODO: write a new grade_documents_v2 node function that filters on
#       result.score == 'yes' AND result.confidence >= CONFIDENCE_THRESHOLD
# HINT: rebuild and recompile the graph to use grade_documents_v2

In [ ]:
# ── Exercise 3 Starter ── Expand the corpus ───────────────────────────────────

new_documents = [
    Document(
        page_content="TODO: replace with your first new document",
        metadata={"source": "my-source", "topic": "my-topic"},
    ),
    # Add 4 more documents on the same topic...
]

# TODO: rebuild the vectorstore with CORPUS + new_documents
# extended_chunks = splitter.split_documents(CORPUS + new_documents)
# extended_store = Chroma.from_documents(extended_chunks, embeddings_model, collection_name="crag-extended")
# extended_retriever = extended_store.as_retriever(search_kwargs={"k": 3})

# TODO: rebuild retrieve() to use extended_retriever
# TODO: recompile the graph
# TODO: run debug_grading() with a question about your new topic
# Expected: all docs score 'yes', web_search stays False

In [ ]:
# ── Exercise 4 Starter ── Persistent checkpointing ────────────────────────────

# from langgraph.checkpoint.memory import MemorySaver
# app_with_memory = graph.compile(checkpointer=MemorySaver())
# config = {"configurable": {"thread_id": "workshop-demo"}}

# TODO: invoke app_with_memory with config for two different questions
# r1 = app_with_memory.invoke(make_initial_state("What is CRAG?"), config=config)
# r2 = app_with_memory.invoke(make_initial_state("How does LangGraph work?"), config=config)

# TODO: inspect the checkpoint state
# snapshot = app_with_memory.get_state(config)
# print(snapshot.values)   # current state values
# print(snapshot.next)     # next node(s) to run (empty at END)

# TODO: print the full state history
# history = list(app_with_memory.get_state_history(config))
# print(f"History entries: {len(history)}")

---

## What's Next?

You now understand the full CRAG pattern: retrieve, grade, optionally correct via web search, generate. Here is where to go deeper.

### Related examples in this repo
- **Example 12 — Basic RAG Notebook** ([`../12-basic-rag-notebook/rag_workbook.ipynb`](../12-basic-rag-notebook/rag_workbook.ipynb)): the standard RAG foundation that CRAG extends — revisit if any of Part 3 felt unfamiliar.
- **Example 16 — RAG Evaluation with RAGAS** ([`../16-rag-eval-notebook/rag_eval_workbook.ipynb`](../16-rag-eval-notebook/rag_eval_workbook.ipynb)): measure your CRAG pipeline on Faithfulness, Answer Relevance, and Context Recall.
- **Example 25 — Adaptive RAG**: dynamically routes between no-retrieval, standard RAG, and CRAG based on query classification.
- **Example 30 — Agentic RAG**: extends CRAG with multi-step reasoning — the agent decides how many retrieval rounds to perform.

### Improve the grader
- **Fine-grained scoring**: add a `confidence` field (Exercise 2) and tune the threshold against a labelled evaluation set
- **Domain-aware grading**: inject domain context into the grade prompt for specialist corpora
- **Batch grading**: evaluate all documents in a single LLM call to reduce latency and API cost

### Improve retrieval
- **Hybrid search**: combine Chroma vector search with BM25 keyword search (`EnsembleRetriever`) for better exact-match recall
- **Re-ranking**: use a cross-encoder (`ContextualCompressionRetriever`) to re-rank candidates before grading
- **HyDE**: embed a hypothetical answer to the query rather than the raw query text for better semantic alignment

### Go to production
- **Checkpointing**: replace `MemorySaver` with `SqliteSaver` or `AsyncPostgresSaver` for durability across process restarts
- **Observability**: set `LANGCHAIN_TRACING_V2=true` to send every node call to LangSmith for dashboard-level visibility
- **Async**: all LangGraph nodes support `async def` — use `ainvoke()` and `astream()` in FastAPI or any async server

### Further reading
- Yan et al. (2024). *Corrective Retrieval Augmented Generation.* https://arxiv.org/abs/2401.15884
- Lewis et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.* https://arxiv.org/abs/2005.11401
- Asai et al. (2023). *Self-RAG: Learning to Retrieve, Generate, and Critique through Self-Reflection.* https://arxiv.org/abs/2310.11511
- LangGraph conceptual guide: https://langchain-ai.github.io/langgraph/concepts/
- LangGraph streaming how-to: https://langchain-ai.github.io/langgraph/how-tos/streaming/

---

*Workshop complete. The self-correcting loop you built here is the foundation of production-grade adaptive RAG systems.*

---

## Exercise Answer Key

Attempt the exercises before reading these solutions. They are one correct approach — yours may differ.

---

### Exercise 1 — Stricter grader

The key change: replace "is this document relevant?" with "does this document contain a specific fact that directly answers the question?". Topic overlap is explicitly ruled out.

**What good output looks like:** The borderline RAG document (which describes RAG in general rather than explaining CRAG's correction mechanism) scores `no`. The dedicated CRAG document scores `yes`.

In [ ]:
# ── Answer 1 — Stricter grader ────────────────────────────────────────────────

strict_grade_prompt_answer = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a rigorous document relevance grader. "
            "Score a document 'yes' ONLY if it contains a specific fact, definition, "
            "or explanation that DIRECTLY and COMPLETELY addresses the question asked. "
            "Topical overlap alone is NOT sufficient — the document must provide the "
            "specific information a reader would need to answer the question. "
            "When in doubt, score 'no'.",
        ),
        ("human", "Question: {question}\n\nDocument:\n{document}"),
    ]
)

test_q = "What is Corrective RAG?"
borderline = "RAG grounds LLM responses in a knowledge base using retrieved documents."
direct = "CRAG improves retrieval by grading each document and falling back to web search when grading fails."

for label, doc in [
    ("Borderline (expected: 'no')", borderline),
    ("Direct (expected: 'yes')", direct),
]:
    result = (strict_grade_prompt_answer | grader).invoke(
        {"question": test_q, "document": doc}
    )
    print(f"{label}: score={result.score!r}")

### Exercise 2 — Confidence threshold

Add `confidence: float` to the Pydantic model and constrain it with `Field(ge=0.0, le=1.0)`. Update `grade_documents` to treat a low-confidence `yes` the same as a `no`.

The threshold (`0.7`) is a hyperparameter — tune it against a labelled evaluation set. Starting at `0.7` is a reasonable default that reduces borderline passes without over-filtering.

In [ ]:
# ── Answer 2 — Confidence threshold ──────────────────────────────────────────
from pydantic import Field


class GradeDocumentsWithConfidenceAnswer(BaseModel):
    """Document relevance score with a calibrated confidence estimate."""

    score: Literal["yes", "no"]
    confidence: float = Field(
        ge=0.0,
        le=1.0,
        description="Confidence in the relevance decision, 0.0 (uncertain) to 1.0 (certain)",
    )


grader_with_confidence = llm.with_structured_output(GradeDocumentsWithConfidenceAnswer)
CONFIDENCE_THRESHOLD = 0.7


def grade_documents_v2(state: CRAGState) -> dict:
    """Grade with confidence — filter docs scoring below the threshold."""
    relevant_docs = []
    needs_web = False

    for doc in state["documents"]:
        result = (grade_prompt | grader_with_confidence).invoke(
            {"question": state["question"], "document": doc.page_content}
        )
        passes = result.score == "yes" and result.confidence >= CONFIDENCE_THRESHOLD
        if passes:
            relevant_docs.append(doc)
        else:
            needs_web = True

    return {"documents": relevant_docs, "web_search": needs_web}


# Quick test — confirm confidence is returned
test_result = (grade_prompt | grader_with_confidence).invoke(
    {
        "question": "What is CRAG?",
        "document": "CRAG grades documents for relevance and falls back to web search.",
    }
)
passes = test_result.score == "yes" and test_result.confidence >= CONFIDENCE_THRESHOLD
print(f"score={test_result.score!r}, confidence={test_result.confidence:.2f}")
print(f"Passes threshold ({CONFIDENCE_THRESHOLD}): {passes}")

### Exercise 3 — Expand the corpus

After rebuilding the vectorstore with new documents, questions on the new topic should return high-scoring retrieved docs and `web_search` should stay `False`. If web search still triggers, check that your documents directly answer the test questions rather than just mentioning the topic.

In [ ]:
# ── Answer 3 — Expanded corpus (ReAct agents + evaluation) ────────────────────

EXTENDED_CORPUS = CORPUS + [
    Document(
        page_content=(
            "ReAct (Reasoning + Acting) is a prompting framework where an LLM alternates "
            "between Thought steps (reasoning about what to do next) and Action steps "
            "(calling tools or APIs). The interleaving grounds reasoning in real observations."
        ),
        metadata={"source": "react-paper", "topic": "react"},
    ),
    Document(
        page_content=(
            "Tool use in LangGraph agents is implemented by adding tool nodes to the graph. "
            "The agent node decides which tool to call; the tool node executes it and returns "
            "results as a ToolMessage appended to the message list in state."
        ),
        metadata={"source": "langgraph-tools", "topic": "tools"},
    ),
    Document(
        page_content=(
            "RAGAS evaluates RAG pipelines on three core metrics: Faithfulness (is the answer "
            "grounded in the retrieved context?), Answer Relevance (does the answer address "
            "the question?), and Context Recall (did retrieval find the necessary facts?)."
        ),
        metadata={"source": "ragas-docs", "topic": "evaluation"},
    ),
    Document(
        page_content=(
            "HyDE (Hypothetical Document Embeddings) improves RAG retrieval by generating "
            "a hypothetical answer to the query first, then embedding that answer for the "
            "similarity search — rather than embedding the raw query directly."
        ),
        metadata={"source": "hyde-paper", "topic": "retrieval"},
    ),
    Document(
        page_content=(
            "Pinecone is a managed cloud vector database. Unlike ChromaDB which runs locally, "
            "Pinecone scales to billions of vectors, supports metadata filtering, namespaces, "
            "and serverless indexes with sub-millisecond query latency at scale."
        ),
        metadata={"source": "pinecone-docs", "topic": "vector-db"},
    ),
]

extended_chunks = splitter.split_documents(EXTENDED_CORPUS)
extended_store = Chroma.from_documents(
    extended_chunks,
    embeddings_model,
    collection_name="crag-extended",
)
extended_retriever = extended_store.as_retriever(search_kwargs={"k": 3})

print(f"Extended vectorstore: {len(extended_chunks)} chunks")
print()

# Verify: questions about new topics should retrieve relevant docs
for q in ["What is ReAct prompting?", "How does RAGAS measure RAG quality?"]:
    docs = extended_retriever.invoke(q)
    print(f"Q: {q}")
    for doc in docs[:2]:
        print(f"  [{doc.metadata.get('topic')}] {doc.page_content[:80]}...")
    print()

### Exercise 4 — Persistent checkpointing

`MemorySaver` stores the full state snapshot after each node in an in-memory dictionary keyed by `thread_id`. Every invocation with the same thread ID adds to the checkpoint history — you can replay or inspect past states.

In production, replace `MemorySaver` with `SqliteSaver` for single-process durability, or `AsyncPostgresSaver` for distributed deployments.

In [ ]:
# ── Answer 4 — Persistent checkpointing ───────────────────────────────────────

try:
    from langgraph.checkpoint.memory import MemorySaver

    app_with_memory = graph.compile(checkpointer=MemorySaver())
    config = {"configurable": {"thread_id": "workshop-demo"}}

    # First invocation
    r1 = app_with_memory.invoke(make_initial_state("What is CRAG?"), config=config)
    print("Run 1 complete.")
    print(f"  web_search: {r1['web_search']}")
    print(f"  answer: {r1['generation'][:120]}...")

    # Inspect checkpoint after first run
    snapshot = app_with_memory.get_state(config)
    print(f"\nCheckpoint after run 1:")
    print(f"  next: {snapshot.next}  (empty list = graph reached END)")
    print(f"  state keys: {list(snapshot.values.keys())}")

    # Second invocation — different question, same thread
    r2 = app_with_memory.invoke(
        make_initial_state("How does LangGraph handle conditional branching?"), config=config
    )
    print(f"\nRun 2 complete.")
    print(f"  web_search: {r2['web_search']}")
    print(f"  answer: {r2['generation'][:120]}...")

    # History shows all checkpoints for this thread
    history = list(app_with_memory.get_state_history(config))
    print(f"\nTotal checkpoint history entries for thread 'workshop-demo': {len(history)}")
    print("(One entry per node execution per run)")

except ImportError as e:
    print(f"MemorySaver not available: {e}")
    print("Upgrade to langgraph>=0.2.0 for checkpointing support.")